In [21]:
from client import *
from common_imports import *

In [22]:
# Add references
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import FunctionTool
from azure.identity import DefaultAzureCredential
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam
from functions import next_visible_event, calculate_observation_cost, generate_observation_report


In [23]:
project_endpoint = MICROSOFT_FOUNDRY
model_deployment = AZURE_OPENAI_DEPLOYMENT_ID

In [24]:
credential = DefaultAzureCredential()

In [25]:
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    model_deployment=model_deployment
)

In [26]:
openai_client = project_client.get_openai_client()

In [27]:
# Define the event function tool
event_tool = FunctionTool(
    name="next_visible_event",
    description="Get the next visible event in a given location.",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "continent to find the next visible event in (e.g. 'north_america', 'south_america', 'australia')",
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    strict=True,
)

In [28]:
# Define the observation cost function tool
cost_tool = FunctionTool(
    name="calculate_observation_cost",
    description="Calculate the cost of an observation based on the telescope tier, number of hours, and priority level.",
    parameters={
        "type": "object",
        "properties": {
            "telescope_tier": {
                "type": "string",
                "description": "the tier of the telescope (e.g. 'standard', 'advanced', 'premium')",
            },
            "hours": {
                "type": "number",
                "description": "the number of hours for the observation",
            },
            "priority": {
                "type": "string",
                "description": "the priority level of the observation (e.g. 'low', 'normal', 'high')",
            },
        },
        "required": ["telescope_tier", "hours", "priority"],
        "additionalProperties": False,
    },
    strict=True,
)

In [29]:
# Define the observation report generation function tool
report_tool = FunctionTool(
    name="generate_observation_report",
    description="Generate a report summarizing an astronomical observation",
    parameters={
        "type": "object",
        "properties": {
            "event_name": {
                "type": "string",
                "description": "the name of the astronomical event being observed",
            },
            "location": {
                "type": "string",
                "description": "the location of the observer",
            },
            "telescope_tier": {
                "type": "string",
                "description": "the tier of the telescope used for the observation (e.g. 'standard', 'advanced', 'premium')",
            },
            "hours": {
                "type": "number",
                "description": "the number of hours the telescope was used for the observation",
            },
            "priority": {
                "type": "string",
                "description": "the priority level of the observation (e.g. 'low', 'normal', 'high')",
            },
            "observer_name": {
                "type": "string",
                "description": "the name of the person who conducted the observation",
            },                   
        },
        "required": ["event_name", "location", "telescope_tier", "hours", "priority", "observer_name"],
        "additionalProperties": False,
    },
    strict=True,
)

In [30]:
# Create a new agent with the function tools
agent = project_client.agents.create_version(
    agent_name="astronomy-agent",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=
            """You are an astronomy observations assistant that helps users find 
            information about astronomical events and calculate telescope rental costs. 
            Use the available tools to assist users with their inquiries.""",
        tools=[event_tool, cost_tool, report_tool],
    ),
)

In [31]:
# Create a thread for the chat session
conversation = openai_client.conversations.create()

In [32]:
# Create a list to hold function call outputs that will be sent back as input to the agent
input_list: ResponseInputParam = []

In [39]:
user_input = "Tell about the eclipse event in north_america and calculate the cost of observing it with a premium telescope for 3 hours at high priority. Also, generate a report for the observation."

In [40]:
# Send a prompt to the agent
openai_client.conversations.items.create(
    conversation_id=conversation.id,
    items=[{"type": "message", "role": "user", "content": user_input}],
)

ConversationItemList(data=[Message(id='msg_d115075b9dbfca7900NYyzz2ZlTCU9jFTi6OdcCjIbm60Gpk1d', content=[ResponseInputText(text='Tell about the eclipse event in north_america and calculate the cost of observing it with a premium telescope for 3 hours at high priority. Also, generate a report for the observation.', type='input_text', prompt_cache_breakpoint=None)], role='user', status='completed', type='message', phase=None)], first_id='msg_d115075b9dbfca7900NYyzz2ZlTCU9jFTi6OdcCjIbm60Gpk1d', has_more=False, last_id='msg_d115075b9dbfca7900NYyzz2ZlTCU9jFTi6OdcCjIbm60Gpk1d', object='list')

In [41]:
# Retrieve the agent's response, which may include function calls
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input=input_list,
)

# Check the run status for failures
if response.status == "failed":
    print(f"Response failed: {response.error}")

In [42]:
# Process function calls
for item in response.output:
    if item.type == "function_call":
        # Retrieve the matching function tool
        function_name = item.name
        result = None
        if item.name == "next_visible_event":
            result = next_visible_event(**json.loads(item.arguments))
        elif item.name == "calculate_observation_cost":
            result = calculate_observation_cost(**json.loads(item.arguments))
        elif item.name == "generate_observation_report":
            result = generate_observation_report(**json.loads(item.arguments))
                
        # Append the output text
        input_list.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=result,
            )
        )

In [43]:
# Send function call outputs back to the model and retrieve a response
if input_list:
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
# Display the agent's response
print(f"AGENT: {response.output_text}")

AGENT: The next visible astronomical event in North America is the **Perseids Meteor Shower** occurring on **August
12th**. 

### Observation Cost
Observing the event with a **premium telescope** for **3 hours** at **high priority** will cost **$1,575.**

### Observation Report
A detailed observation report for this event has been generated. You can access the report here: 
`report_perseids_meteor_shower_2026-07-13_1255.txt`.

Let me know if you need additional assistance!

In [ ]:
 # Delete the agent when done
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Deleted agent.")

Deleted agent.